# OCTA MAE — Phase 1 Pretraining

Trénuje ViT-Small encoder na jednotlivých OCTA snímkach (SVP + DCP ako samostatné vzorky).


## Imports + Cesty


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "scripts")
from octa_mae_core import *

ENCODER_ROOT = Path().resolve().parent 
RESULTS      = Path().resolve() / "results"  

EXCEL_PATH = ENCODER_ROOT / "data" / "master_excels" / "master_table.xlsx"
DATA_ROOT  = ENCODER_ROOT / "data"

print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM   : {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")
print(f"wandb  : {'available' if WANDB_AVAILABLE else 'not installed — logging disabled'}")
print(f"Excel  : {EXCEL_PATH}")
print(f"Data root: {DATA_ROOT}")

---
## Phase 1 Baseline
Štandardný shuffle sampler. Berie všetky snímky kde `use_for_mae==1`.

In [ ]:
p1_cfg = Phase1Config(
    excel_path  = str(EXCEL_PATH),
    data_root   = str(DATA_ROOT),
    output_base = str(RESULTS / "phase1" / "baseline"),

    # Tréning
    epochs        = 250,
    batch_size    = 64,
    lr            = 1.5e-4,
    warmup_epochs = 40,
    save_every    = 25,

    wandb_project  = "octa-mae",
    wandb_entity   = None,
    wandb_run_name = None,
)

print("Spúšťam Phase 1 Baseline...")
p1_encoder, p1_encoder_path = train_phase1(p1_cfg)

print(f"\n✅ Phase 1 Baseline hotovo!")
print(f"   Encoder uložený: {p1_encoder_path}")

---
## Phase 1 Balanced
Každý batch obsahuje 50% SVP + 50% DCP vzorky. DCP sa oversampluje ak je ich menej.

In [ ]:
# ── Config — zmeň čo potrebuješ ───────────────────────────────────────────────
p1b_cfg = Phase1BalancedConfig(
    excel_path  = EXCEL_PATH,
    data_root   = DATA_ROOT,
    output_base = str(RESULTS / "phase1" / "balanced"),

    # Tréning
    epochs        = 250,
    batch_size    = 64,
    lr            = 1.5e-4,
    warmup_epochs = 40,
    save_every    = 25,

    wandb_project  = "octa-mae",
    wandb_entity   = None,
    wandb_run_name = None,
)

print("Spúšťam Phase 1 Balanced...")
p1b_encoder, p1b_encoder_path = train_phase1_balanced(p1b_cfg)

print(f"\n✅ Phase 1 Balanced hotovo!")
print(f"   Encoder uložený: {p1b_encoder_path}")